# HelixIR — Live Demo

**Runtime → Change runtime type → T4 GPU** before running.

This notebook walks through all four HelixIR capabilities:
1. Static JAXPR analysis + roofline classification
2. Auto-sharding code generation (Megatron column/row parallelism)
3. Backward-pass FLOP analysis via `jax.vjp`
4. Real GPU benchmarks (JIT vs no-JIT, attention scaling, GPT block throughput)

In [ ]:
# Cell 1 — install
!pip install -q git+https://github.com/HackathonGroupMulti/HelixIR.git
print('Done. If this is your first install, use Runtime → Restart session, then re-run from Cell 2.')

In [ ]:
# Cell 2 — imports + sanity check
import jax
import jax.numpy as jnp
import helix

print('JAX version :', jax.__version__)
print('HelixIR     :', helix.__version__)
print('Devices     :', jax.devices())

## 1 — Static analysis: roofline + advisor passes

In [ ]:
# Cell 3 — LLaMA-style transformer block analysis
key = jax.random.PRNGKey(0)
B, S, D, H = 4, 512, 1024, 16
head_dim = D // H

def transformer_tiny(x, wq, wk, wv, wo, w1, w2, norm):
    rms = jnp.sqrt(jnp.mean(x**2, axis=-1, keepdims=True) + 1e-6)
    h   = (x / rms) * norm
    q   = jnp.einsum('bsd,dh->bsh', h, wq)
    k   = jnp.einsum('bsd,dh->bsh', h, wk)
    v   = jnp.einsum('bsd,dh->bsh', h, wv)
    s   = jnp.einsum('bqd,bkd->bqk', q, k) * (head_dim ** -0.5)
    a   = jax.nn.softmax(s, axis=-1)
    out = jnp.einsum('bqk,bkd->bqd', a, v)
    h2  = jax.nn.silu(jnp.einsum('bsd,dh->bsh', out + x, w1))
    return jnp.einsum('bsd,dh->bsh', h2, w2)

args = (
    jax.random.normal(key, (B, S, D)),   # x
    jax.random.normal(key, (D, D)),       # wq
    jax.random.normal(key, (D, D)),       # wk
    jax.random.normal(key, (D, D)),       # wv
    jax.random.normal(key, (D, D)),       # wo
    jax.random.normal(key, (D, D * 4)),   # w1  (gate/up, fan-out)
    jax.random.normal(key, (D * 4, D)),   # w2  (down, fan-in)
    jnp.ones(D),                          # norm
)

report = helix.analyze(transformer_tiny, *args)

r = report['roofline']
print(f'Ops          : {report["num_ops"]}')
print(f'Total FLOPs  : {report["total_flops"]/1e9:.2f} GFLOPs')
print(f'Total bytes  : {report["total_bytes"]/1e6:.1f} MB')
print(f'Ridge point  : {r.ridge_point:.1f} FLOP/byte')
print(f'Compute-bound: {len(r.compute_bound_ops)} ops')
print(f'BW-bound     : {len(r.bandwidth_bound_ops)} ops')
print()
for p in report['passes']:
    print(f'[{p.pass_name}] {p.summary}')
    for rec in p.recommendations[:3]:
        print(f'  • {rec["message"]}')

## 2 — Auto-sharding: Megatron column/row parallelism

In [ ]:
# Cell 4 — generate sharding plan + backward analysis
plan = helix.generate_sharding(
    transformer_tiny, *args,
    mesh_shape=(2, 4),
    axis_names=('batch', 'model'),
    arg_names=['x','wq','wk','wv','wo','w1','w2','norm'],
)
print(plan)   # human-readable table
print()
print('=== Generated code (paste into training script) ===')
print(plan.code)

## 3 — Backward-pass analysis

In [ ]:
# Cell 5 — forward vs backward FLOP breakdown
full = helix.analyze_full(transformer_tiny, *args)
helix.print_full_report(full, 'transformer_tiny')
print(f'bwd / fwd FLOP ratio : {full["bwd_fwd_flop_ratio"]:.2f}×  (theory ≈ 3×)')

## 4 — GPU benchmarks

All timings use `jax.block_until_ready()` so we measure real device time, not
dispatch latency.  10 warm-up + 50 timed iterations.

In [ ]:
# Cell 6 — RMSNorm: JIT vs no-JIT
import time

from helix.kernels.rmsnorm import rmsnorm_ref

x_norm = jax.random.normal(key, (32, 2048))
w_norm = jnp.ones(2048)
rmsnorm_jit = jax.jit(rmsnorm_ref)

def _time_fn(fn, *args, warmup=10, iters=50):
    for _ in range(warmup):
        jax.block_until_ready(fn(*args))
    t0 = time.perf_counter()
    for _ in range(iters):
        jax.block_until_ready(fn(*args))
    return (time.perf_counter() - t0) / iters * 1e3  # ms

t_ref = _time_fn(rmsnorm_ref, x_norm, w_norm)
t_jit = _time_fn(rmsnorm_jit, x_norm, w_norm)
print(f'RMSNorm no-JIT : {t_ref:.3f} ms')
print(f'RMSNorm JIT    : {t_jit:.3f} ms')
print('(Memory-bound op — JIT removes dispatch overhead but HBM bandwidth is the bottleneck)')

In [ ]:
# Cell 7 — Attention: O(S²) scaling
from helix.kernels.attention import attention_ref

attn_jit = jax.jit(attention_ref)
print(f'{"seq_len":>10}  {"latency (ms)":>14}')
print('-' * 28)
for seq_len in [128, 256, 512, 1024]:
    q = jax.random.normal(key, (2, seq_len, 8, 64))
    k = jax.random.normal(key, (2, seq_len, 8, 64))
    v = jax.random.normal(key, (2, seq_len, 8, 64))
    t = _time_fn(attn_jit, q, k, v)
    print(f'{seq_len:>10}  {t:>14.3f}')
print('\nExpected: ~4× slowdown per 2× seq (O(S²) attention)')

In [ ]:
# Cell 8 — GPT block end-to-end throughput
from helix.benchmark.workloads.transformer import gpt_block, make_inputs, flop_count

fn, params = gpt_block(batch=4, seq=512, dim=1024, heads=16)
inputs     = make_inputs(batch=4, seq=512, dim=1024)
fn_jit     = jax.jit(fn)

latency_ms = _time_fn(fn_jit, inputs, params)
flops      = flop_count(batch=4, seq=512, dim=1024)
tflops     = flops / (latency_ms * 1e-3) / 1e12

print(f'GPT block (B=4 S=512 D=1024)')
print(f'  Latency   : {latency_ms:.3f} ms')
print(f'  Throughput: {tflops:.2f} TFLOPS')
print(f'  T4 peak   : 65 TFLOPS FP16')
print(f'  Utilization: {tflops/65*100:.1f}%')
print('\n(Low utilization = single-batch inference; Flash Attn + sharding are the levers)')

## Next steps

- Switch to an **A100 / H100 runtime** to enable the fused Pallas kernels (`rmsnorm_pallas`, `rope_pallas`, `flash_attention`)
- Paste the generated sharding code into a multi-device training script and call `jax.jit` — JAX inserts the collective ops automatically
- Use `helix.analyze_full` to identify which backward ops dominate memory and apply `jax.checkpoint` to the targets flagged by `CheckpointAdvisorPass`